___
<img style="float: right; margin: 15px 15px 15px 15px;" src="https://beehiiv-images-production.s3.amazonaws.com/uploads/asset/file/cecbccba-6358-476e-9fd8-e2807de9f220/Frame_118.png?t=1693044751" width="300px" height="180px" />


# <font color= #bbc28d> **Hugging Face: Model Sharing, Inference & Deployment** </font>
#### <font color= #2E9AFE> `Tarea 6 – Text Mining`</font>
- <Strong> Sofía Maldonado, Diana Valdivia & Viviana Toledo </Strong>
- <Strong> Fecha </Strong>: 13/11/2025 

___

<p style="text-align:right;"> Imagen recuperada de: https://beehiiv-images-production.s3.amazonaws.com/uploads/asset/file/cecbccba-6358-476e-9fd8-e2807de9f220/Frame_118.png?t=1693044751</p>

In [4]:
# Cargar Librerías
from huggingface_hub import create_repo, Repository, notebook_login
from transformers import AutoModelForSequenceClassification, AutoTokenizer

# <font color= #bbc28d> **1. Cargar a Hugging Face Hub** </font>

#### **¿Qué es?**
El Hub de Hugging Face funciona como una plataforma centralizada para compartir y versionar modelos, datasets y espacios.
Es similar a GitHub, pero enfocado en inteligencia artificial: puedes subir tus modelos entrenados, mantener versiones y compartirlos fácilmente.

### <font color= #bbc28d> **Pre-requisitos** </font>

Antes de empezar, asegúrate de tener lo siguiente configurado correctamente:

1. **Cuenta en `Hugging Face`**  
   - Crea una cuenta.
   - Ve a tu perfil → *Settings* → *Access Tokens* → Crea un nuevo token con el rol `write`.
        - Read access to contents of all repos under your personal namespace
        - Write access to contents/settings of all repos under your personal namespace

2. **Instalar Git y Git LFS**

    Para poder manejar con repositorios dentro de Hugging Face es necesario tener instalado dos cosas:
   - Descarga **Git** desde [https://git-scm.com/downloads](https://git-scm.com/downloads).
   - Descarga **Git LFS** desde [https://git-lfs.github.com/](https://git-lfs.github.com/).

   Luego abre tu terminal (Anaconda Prompt o CMD) y ejecuta:

   ```bash
   git lfs install
   ```
   **NOTA**: Esto dentro del ambiente del proyecto.

3. **Instalar dependencias en tu entorno:**
   ```bash
   pip install huggingface_hub transformers ipywidgets
   ```

   Versiones sugeridas:
   - `huggingface_hub >= 0.24.0`
   - `transformers >= 4.43.0`
   - `git >= 2.40`
   - `git-lfs >= 3.5.0`

In [5]:
# Instalar librerías necesarias
# !pip install -q transformers==4.45.2 huggingface_hub==0.26.2 ipywidgets

  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 5.49.1 requires huggingface-hub<2.0,>=0.33.5, but you have huggingface-hub 0.26.2 which is incompatible.


### <font color= #bbc28d> **Autentificación** </font>
Antes de subir cualquier archivo o modelo, debemos iniciar sesión con nuestro token personal. Esto permite a Hugging Face reconocer qué usuario está subiendo el modelo.

**NOTA**: Es recomendable tener un respaldo del token ya que solo es visible la primera vez que es creado.

In [11]:
# Hacer login con el token de Hugging Face
notebook_login()  

### <font color= #bbc28d> **Guardar el Modelo** </font>
Antes de subir un modelo al Hub necesitamos un modelo que subir, para esto cargaremos uno preentrenado desde Hugging Face para usarlo como ejemplo.

- **`model_name = "distilbert-base-uncased"`**: define el modelo base, una versión ligera de BERT.  
- **`AutoTokenizer.from_pretrained(model_name)`**: carga el tokenizador asociado, encargado de convertir texto en tokens numéricos.  
- **`AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)`**: carga el modelo preentrenado y lo adapta para una tarea de clasificación binaria.

**NOTA**: Este modelo puede ser uno creado desde cero, esto es solo para concentrarnos en subir modelos al Hub.


In [12]:
# Cargamos un modelo preentrenado simple
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


### <font color= #bbc28d> **Subir el modelo al Hub** </font>
### **1. Configuración de identidad de Git**
Se define el **nombre de usuario** y el **correo electrónico** asociados a Git mediante los comandos:
- `!git config --global user.name`
- `!git config --global user.email`

Esto permite registrar quién realiza los **commits** dentro del historial del repositorio y vincular los cambios con tu identidad de desarrollador.

`NOTA`: Estas credenciales son las de Hugging Face, aunque se maneje con Git, no hacen referencia a ellas.


In [4]:
!git config --global user.name "pipo1313"
!git config --global user.email "pipochatgpt@gmail.com"

### **2. Creación del repositorio remoto**
Con la función **`create_repo()`**, se crea un **nuevo repositorio remoto** en la cuenta de Hugging Face.  
- Si el repositorio ya existe, el argumento `exist_ok=True` evita errores y permite reutilizarlo.  
Este repositorio actuará como el **espacio en la nube** donde se almacenarán los archivos del modelo y el tokenizer.

In [17]:
# Nombre del repositorio
repo_name = "distilbert-demo-model"

# Crea un nuevo repositorio en tu cuenta
create_repo(repo_name, exist_ok=True)

RepoUrl('https://huggingface.co/aiko-isnt-ded/distilbert-demo-model', endpoint='https://huggingface.co', repo_type='model', repo_id='aiko-isnt-ded/distilbert-demo-model')

### **3. Clonación o conexión con el repositorio local**
Se utiliza la clase **`Repository()`** para conectar el repositorio remoto con una carpeta local.  
Aquí intervienen dos parámetros importantes:
- `local_dir=repo_name`: crea (o usa) una carpeta local con el nombre del repositorio donde se guardarán los archivos del modelo.  
- `clone_from=f"usuario/nombre_repo"`: clona el repositorio remoto de Hugging Face dentro del directorio local.  

Esto permite trabajar localmente con los archivos del modelo y luego sincronizarlos con el Hub mediante Git.

In [ ]:
repo = Repository(local_dir=repo_name, clone_from=f"pipo1313/{repo_name}")

### **4. Guardado del modelo y del tokenizer**
Se guardan los archivos del modelo y del tokenizer usando:
- `model.save_pretrained(repo_name)`
- `tokenizer.save_pretrained(repo_name)`



In [26]:
# Guardar el modelo y tokenizer dentro de esa carpeta
model.save_pretrained(repo_name)
tokenizer.save_pretrained(repo_name)

('distilbert-demo-model\\tokenizer_config.json',
 'distilbert-demo-model\\special_tokens_map.json',
 'distilbert-demo-model\\vocab.txt',
 'distilbert-demo-model\\added_tokens.json',
 'distilbert-demo-model\\tokenizer.json')

### **5. Envío (push) de los archivos al Hub**
Una vez guardados los archivos, se realiza la subida con:
- `repo.push_to_hub(commit_message="mensaje")`

Esta función:
- Crea un **commit** que registra los cambios en el repositorio local.
- Luego realiza un **push** al repositorio remoto en Hugging Face Hub.  
El argumento `commit_message` describe la acción que se está subiendo.

In [27]:
# El commit se sube a tu repo en Hugging Face usando tu token de autenticación.
repo.push_to_hub(commit_message="Subida inicial de modelo demo")

# <font color= #bbc28d> **2. Utilizar from_pretrained con modelos personalizados** </font>

Una vez que el modelo ha sido subido al **Hugging Face Hub**, podemos cargarlo fácilmente desde cualquier entorno.

- **`AutoModelForSequenceClassification.from_pretrained("pipo1313/distilbert-demo-model")`**: descarga el modelo directamente desde tu repositorio en el Hub.  
    - Usuario/nombre_repo
- **`AutoTokenizer.from_pretrained("pipo1313/distilbert-demo-model")`**: carga el tokenizador asociado al modelo, garantizando que el texto se procese con el mismo formato que durante el entrenamiento.
    - Usuario/nombre_repo


In [ ]:
# Carga del modelo subido
model = AutoModelForSequenceClassification.from_pretrained("pipo1313/distilbert-demo-model")
tokenizer = AutoTokenizer.from_pretrained("pipo1313/distilbert-demo-model")

### <font color= #bbc28d> **Hacer una predicción de prueba** </font>
Para comprobar que el modelo cargado funciona correctamente, realizamos una predicción con una frase de ejemplo.

- **`tokenizer("Hugging Face is awesome!", return_tensors="pt")`**: convierte el texto en tensores de PyTorch.
- **`model(**inputs)`**: pasa los datos tokenizados al modelo y obtiene las salidas (logits).  
- **`outputs.logits`**: muestra las puntuaciones sin normalizar que el modelo asigna a cada clase.

In [29]:
# Hacer una predicción de prueba
inputs = tokenizer("Hugging Face is awesome!", return_tensors="pt")
outputs = model(**inputs)
print(outputs.logits)

tensor([[-0.0192,  0.0619]], grad_fn=<AddmmBackward0>)


# <font color= #bbc28d> **3. Exportar a ONNX o TorchScript** </font>

Para desplegar modelos a entornos de ejecución, podemos exportar los modelos a un formato estándar que pueda ser cargado y ejecutado en hardware especializado. Algunos de estos formatos son ONNX (Open Neural Network eXchange) y TorchScript.

### <font color= #bbc28d> &ensp; • **ONNX (Open Neural Network eXchange)** </font>

Estándar abierto que define un **formato común para representar modelos de Deep Learning**. Cuando un modelo se exporta al formato ONNX, se construye un grafo computacional que **representa el flujo de datos a través de la red neuronal**. Esto permite que modelos entrenados en PyTorch puedan ser exportados en ONNX e importados en TensorFlow, o viceversa. 

### <font color= #66b0b0> &emsp; &nbsp; **Exportar a ONNX** </font>

Primero, tenemos que instalar algunas dependencias:

&nbsp; `pip install transformers[onnx]`

In [ ]:
# !pip install transformers[onnx]

Para exportar un modelo que está guardado de forma local, los **archivos de pesos y tokenizador del modelo deben de estar almacenados en un directorio**. Para esto, utilizaremos el modelo que cargamos hace un ratito. 

In [38]:
# Load tokenizer and PyTorch weights form the Hub
pt_model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased")
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

# Save to disk
tokenizer.save_pretrained("local-pt-checkpoint")            # It's being stored in local-pt-checkpoint
pt_model.save_pretrained("local-pt-checkpoint")

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Es importante notar que el modelo a exportar debe de tener la terminación **.bin para modelos de PyTorch o .h5 para modelos de Tensorflow**. En este caso, como el modelo es de HuggingFace, primero lo convertiremos en un archivo compatible.

In [39]:
from safetensors.torch import load_file
import torch

weights = load_file("local-pt-checkpoint/model.safetensors")
torch.save(weights, "local-pt-checkpoint/pytorch_model.bin")

Después de guardar el tokenizador y pesos de los modelos en la carpeta `local-pt-checkpoint`, corremos el siguiente comando en terminal en la carpeta donde esté almacenada la ruta. 

&nbsp; `python -m transformers.onnx --model=local-pt-checkpoint onnx/`

Finalmente, el modelo fue exportado a formato ONNX con éxito.

### <font color= #bbc28d> &ensp; • **TorchScript** </font>

TorchScript es una manera de **crear modelos serializables y optimizables a partir del código de PyTorch**. Este formato permite exportar modelos Transformer para que puedan **ejecutarse en entornos diferentes al de los programas basados Python**. Lo anterior requiere:

- Instanciar el modelo con la bandera torchscript
- Un forward-pass con entradas ficticias (dummy inputs)

### <font color= #66b0b0> &emsp; &nbsp; **Definir el Modelo** </font>

Para iniciar, entrenaremos un modelo Transformer tipo Bert. Esto requiere definir las variables de entrada y obtener los tokens de ellas:

In [41]:
from transformers import BertModel, BertTokenizer, BertConfig

enc = BertTokenizer.from_pretrained("bert-base-uncased")

# Tokenizing input text
text = "[CLS] Who was Jim Henson ? [SEP] Jim Henson was a puppeteer [SEP]"
tokenized_text = enc.tokenize(text)

# Masking one of the input tokens
masked_index = 8
tokenized_text[masked_index] = "[MASK]"
indexed_tokens = enc.convert_tokens_to_ids(tokenized_text)
segments_ids = [0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1]

Posteriormente, se crea el **dummy input requerido para TorchScript:**

In [42]:
# Creating a dummy input
tokens_tensor = torch.tensor([indexed_tokens])
segments_tensors = torch.tensor([segments_ids])
dummy_input = [tokens_tensor, segments_tensors]

Y se inicializa el modelo con la **bandera de TorchScript:**

In [43]:
# Initializing the model with the torchscript flag
# Flag set to True even though it is not necessary as this model does not have an LM Head.
config = BertConfig(
    vocab_size_or_config_json_file=32000,
    hidden_size=768,
    num_hidden_layers=12,
    num_attention_heads=12,
    intermediate_size=3072,
    torchscript=True,
)

Después de cumplir con los requerimientos de TorchScript, se ejecuta el modelo y **se aplica una "trace" para empaquetar el modelo**.

In [44]:
# Instantiating the model
model = BertModel(config)

# The model needs to be in evaluation mode
model.eval()

# If you are instantiating the model with *from_pretrained* you can also easily set the TorchScript flag
model = BertModel.from_pretrained("bert-base-uncased", torchscript=True)

# Creating the trace
traced_model = torch.jit.trace(model, [tokens_tensor, segments_tensors])
torch.jit.save(traced_model, "traced_bert.pt")

El modelo se guardó con el nombre `traced_bert.pt`.

### <font color= #66b0b0> &emsp; &nbsp; **Cargar el Modelo** </font>

Después de aplicar el TorchScript, podemos cargarlo a partir del trace generado:

In [45]:
loaded_model = torch.jit.load("traced_bert.pt")
loaded_model.eval()

all_encoder_layers, pooled_output = loaded_model(*dummy_input)

Estas son las capas del modelo almacenado:

In [51]:
all_encoder_layers

tensor([[[-2.5689e-01, -7.3597e-03, -8.9147e-02,  ..., -1.3546e-01,
           2.3597e-01,  2.4208e-01],
         [-5.8262e-01,  3.1923e-01, -2.8020e-01,  ...,  1.0413e-01,
           1.7953e-01, -4.7086e-01],
         [-3.0671e-01, -2.3213e-01, -1.5938e-01,  ...,  7.0993e-02,
           1.4761e-01,  2.7529e-01],
         ...,
         [ 2.0549e-01, -1.6316e-02, -7.1123e-05,  ..., -1.3032e-01,
           6.1008e-01,  4.2999e-01],
         [-4.9530e-01, -4.6195e-01, -2.9027e-01,  ...,  6.3559e-01,
           6.2100e-01,  1.0318e-01],
         [ 8.2051e-01,  1.8250e-01, -1.1302e-01,  ...,  1.5103e-01,
          -7.6513e-01, -1.9481e-02]]], grad_fn=<NativeLayerNormBackward0>)

Y el output al evaluar el modelo con nuestro input:

In [48]:
pooled_output

tensor([[-4.9859e-01, -1.6913e-01,  8.3044e-01,  7.2490e-02, -4.8807e-01,
         -9.1258e-02,  5.1964e-01,  1.2615e-01,  7.3988e-01, -9.9609e-01,
          3.7945e-01, -5.8106e-01,  9.5275e-01, -6.8154e-01,  7.0220e-01,
         -2.4374e-01,  9.2702e-02, -3.1205e-01,  2.3801e-01, -3.3289e-01,
          2.2093e-01, -3.1050e-01,  7.1334e-01,  6.8511e-02,  1.5540e-01,
         -7.6150e-01, -2.8993e-01,  7.5329e-01,  8.3334e-01,  5.7146e-01,
         -3.3142e-01,  1.4806e-01, -9.5029e-01, -9.6337e-02,  7.7407e-01,
         -9.3747e-01, -3.0847e-02, -4.7886e-01,  4.4816e-02,  1.1357e-02,
         -6.1463e-01,  2.0118e-01,  9.0852e-01, -6.6917e-01, -3.0572e-01,
         -2.4733e-01, -6.9975e-01,  9.1703e-02, -6.5932e-01, -8.4444e-01,
         -7.1574e-01, -8.7206e-01,  5.5364e-02,  1.0259e-01,  2.0758e-01,
          4.1869e-01, -2.2090e-01,  6.6650e-02,  3.9337e-02, -3.4385e-01,
         -4.0217e-01,  3.1609e-02,  6.8003e-01, -7.3265e-01, -7.0735e-01,
         -8.4914e-01, -3.4449e-02, -7.

# <font color= #bbc28d> **4. Servir modelos con FastAPI o Endpoints de Inferencia** </font>